# 05 — Economic context and moving-average signals

This notebook returns to the broader question behind the original research: how do the S&P 500 and major economic series move over time? It also illustrates a 50/200-session moving-average rule with a one-session signal delay and a simplified transaction cost.

The economic plots are **descriptive**. GDP, CPI, and unemployment CSVs are indexed by observation period, which can be earlier than the actual release date. They are not used as forecasting inputs here.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Download a sample if no local CSV is available (internet required).
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
import matplotlib.pyplot as plt
from sp500_research.strategies import illustrative_sma_strategy
result=illustrative_sma_strategy(prices.Close, short=50, long=200,
                                  cost_per_turnover=0.001)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(result.index,result.buy_hold_cumulative,label='Index buy & hold, idealized')
ax.plot(result.index,result.sma_cumulative,label='50/200 SMA, simplified cost')
ax.set(title='Illustrative history; not an executable trading policy',ylabel='Growth of $1')
ax.legend();plt.tight_layout();plt.show()

In [ ]:
from pathlib import Path
from sp500_research.macro import read_fred_observations, observation_period_overlay
DATA=Path('../data')
for code in ('GDP','CPIAUCSL','UNRATE'):
    path=DATA/f'{code}.csv'
    if not path.exists():
        print('Optional observational CSV not available:',path)
        continue
    series=read_fred_observations(path)
    descriptive=observation_period_overlay(prices,series)
    print(code,'observational rows:',len(descriptive))
    # Rescale both series for plotting; these are not model features.
    if len(descriptive)<5:continue
    a=descriptive.iloc[:,0]/descriptive.iloc[:,0].iloc[0]
    b=descriptive.iloc[:,1]/descriptive.iloc[:,1].iloc[0]
    fig,ax=plt.subplots(figsize=(9,3))
    ax.plot(descriptive.index,a,label='S&P normalized level')
    ax.plot(descriptive.index,b,label=f'{code} period-labeled normalized level')
    ax.set(title=f'DESCRIPTIVE: {code} & market level (not release-date aligned)')
    ax.legend();plt.tight_layout();plt.show()

### A note on interpretation

Two long-running series can rise together without one predicting the other. The moving-average example is also deliberately simplified: it leaves out financing, slippage, and the mechanics of trading an index exposure. It should be read as a historical illustration, not a tested trading strategy.